# Air Quality Category Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `aqi_category`

## 2 · Project Overview

We classify **air quality** into 4 categories (Good, Moderate, Unhealthy for Sensitive Groups, Unhealthy) based on pollutant concentrations (PM2.5, PM10, NO₂, SO₂, CO, O₃) and weather conditions.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given pollutant concentrations and weather data, classify the air quality index (AQI) category.

## 5 · Why This Project Matters

- **Air quality monitoring** directly impacts public health decisions.
- AQI prediction enables health advisories and activity planning.
- Understanding pollutant interactions improves environmental policy.
- Demonstrates ordinal multiclass classification in environmental science.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | pm25, pm10, no2, so2, co, o3, temperature, humidity |
| **Target** | aqi_category (4 classes: Good, Moderate, Unhealthy_Sensitive, Unhealthy) |
| **Class balance** | Varies by pollutant levels |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "aqi_category"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: aqi_category
Save dir: E:\Github\Machine-Learning-Projects\Classification\Air Quality Category Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 air quality readings with pollutant concentrations and AQI category.

In [4]:
np.random.seed(SEED)
n = 8000
pm25 = np.round(np.random.lognormal(2.5, 0.8, n).clip(1, 500), 1)
pm10 = np.round(pm25 * np.random.uniform(1.2, 2.5, n) + np.random.normal(0, 10, n), 1).clip(1, 600)
no2 = np.round(np.random.lognormal(2.8, 0.6, n).clip(1, 200), 1)
so2 = np.round(np.random.exponential(10, n).clip(0.5, 100), 1)
co = np.round(np.random.lognormal(0.5, 0.5, n).clip(0.1, 30), 2)
o3 = np.round(np.random.lognormal(3.2, 0.5, n).clip(5, 300), 1)
temperature = np.round(np.random.normal(22, 8, n).clip(-10, 45), 1)
humidity = np.round(np.random.uniform(20, 95, n), 1)

# AQI primarily driven by PM2.5
aqi_score = pm25
aqi_category = np.where(aqi_score <= 12, "Good",
               np.where(aqi_score <= 35, "Moderate",
               np.where(aqi_score <= 55, "Unhealthy_Sensitive", "Unhealthy")))

for cat in ["Good", "Moderate", "Unhealthy_Sensitive", "Unhealthy"]:
    if (aqi_category == cat).sum() < 20:
        idxs = np.random.choice(n, 20, replace=False)
        aqi_category[idxs[:max(0, 20-(aqi_category==cat).sum())]] = cat

df = pd.DataFrame({"pm25": pm25, "pm10": pm10, "no2": no2, "so2": so2,
                    "co": co, "o3": o3, "temperature": temperature,
                    "humidity": humidity, "aqi_category": aqi_category})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['aqi_category'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 9)
Class balance:
aqi_category
Good                   0.498
Moderate               0.408
Unhealthy_Sensitive    0.064
Unhealthy              0.031
Name: proportion, dtype: float64


,pm25,pm10,no2,so2,co,o3,temperature,humidity,aqi_category
0,18.1,9.6,7.2,10.3,2.95,36.4,24.7,69.5,Moderate
1,10.9,9.0,16.5,5.5,1.68,24.3,18.7,30.1,Good
2,20.5,39.9,19.0,16.7,1.37,9.7,34.3,53.4,Moderate
3,41.2,81.6,9.0,5.6,0.83,7.3,14.7,42.2,Unhealthy_Sensitive
4,10.1,18.8,52.7,8.5,1.16,27.5,3.6,45.3,Good


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 9)

Missing values:
pm25            0
pm10            0
no2             0
so2             0
co              0
o3              0
temperature     0
humidity        0
aqi_category    0
dtype: int64

Duplicate rows: 0

Dtypes:
pm25            float64
pm10            float64
no2             float64
so2             float64
co              float64
o3              float64
temperature     float64
humidity        float64
aqi_category     object
dtype: object

Target 'aqi_category' confirmed. Value counts:
aqi_category
Good                   3980
Moderate               3264
Unhealthy_Sensitive     511
Unhealthy               245
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["pm25", "pm10", "no2", "so2", "co", "o3"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Pollutant Distributions by AQI Category", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Pollutant Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `aqi_category`.

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ["Good", "Moderate", "Unhealthy_Sensitive", "Unhealthy"]
df[TARGET].value_counts().reindex(order).plot(kind="bar", ax=ax,
    color=["green", "gold", "orange", "red"], edgecolor="black")
ax.set_title("AQI Category Distribution")
plt.tight_layout()
plt.show()
print(f"Category counts:\n{df[TARGET].value_counts()}")

Category counts:
aqi_category
Good                   3980
Moderate               3264
Unhealthy_Sensitive     511
Unhealthy               245
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 8), Test: (1600, 8)
Train class distribution:
aqi_category
0    0.498
1    0.408
3    0.064
2    0.031
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: Target `aqi_category` encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: Varies by pollutant distribution (log-normal).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["pm_ratio"] = X_train["pm25"] / (X_train["pm10"] + 1)
X_test["pm_ratio"] = X_test["pm25"] / (X_test["pm10"] + 1)

X_train["total_pollutant"] = X_train["pm25"] + X_train["no2"] + X_train["so2"] + X_train["co"] * 10
X_test["total_pollutant"] = X_test["pm25"] + X_test["no2"] + X_test["so2"] + X_test["co"] * 10

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'pm_ratio', 'total_pollutant']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.9856
F1       : 0.9854

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       796
           1       0.98      0.99      0.99       653
           2       0.94      0.98      0.96        49
           3       0.94      0.86      0.90       102

    accuracy                           0.99      1600
   macro avg       0.96      0.96      0.96      1600
weighted avg       0.99      0.99      0.99      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
BaggingClassifier           1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.083123
DecisionTreeClassifier      1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.024244
RandomForestClassifier      1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    0.537006
LGBMClassifier              1.000000           1.000000  1.000000  1.000000   1.000000  1.000000    3.233073
XGBClassifier               0.999375           0.997549  0.999995  0.999377   0.999387  0.999375    0.211270
CatBoostClassifier          0.998750           0.995098  0.999999  0.998745   0.998754  0.998750    4.661911
ExtraTreesClassifier        0.991250           0.970142  0.999868  0.991200   0.991236  0.9912

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: lgbm
Accuracy: 1.0000
F1: 1.0000


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.9919  (3.1s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[130]	valid_0's multi_logloss: 0.000298678
LightGBM F1: 1.0000  (3.5s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               1.0000  1.0000     1.0000  1.0000
FLAML                  1.0000  1.0000     1.0000  1.0000
CatBoost               0.9919  0.9919     0.9919  0.9919
Logistic Regression    0.9856  0.9854     0.9854  0.9856

Top 3 models: ['LightGBM', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  LightGBM:
    Accuracy : 1.0000
    F1       : 1.0000
    Precision: 1.0000
    Recall   : 1.0000

  FLAML:
    Accuracy : 1.0000
    F1       : 1.0000
    Precision: 1.0000
    Recall   : 1.0000

  CatBoost:
    Accuracy : 0.9919
    F1       : 0.9919
    Precision: 0.9919
    Recall   : 0.9919


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: LightGBM

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       796
           1       1.00      1.00      1.00       653
           2       1.00      1.00      1.00        49
           3       1.00      1.00      1.00       102

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600


Total errors: 0 / 1600 (0.0%)


## 25 · Interpretation and Business Insight

**Key findings:**
- **PM2.5** is the primary AQI driver (by EPA definition).
- **PM10** and **NO₂** are correlated with PM2.5.
- **Temperature** and **humidity** affect pollutant dispersion.
- **PM ratio** (PM2.5/PM10) indicates combustion vs. dust sources.

**Business takeaway:** PM2.5 monitoring is the most critical single measurement for air quality assessment.

## 26 · Limitations

1. AQI categories are directly derived from PM2.5 (near-deterministic).
2. No geographic or temporal features.
3. Missing wind speed and direction.
4. No source attribution (traffic, industrial, natural).
5. Simplified EPA breakpoints.

## 27 · How to Improve This Project

1. Use real EPA air quality monitoring data.
2. Add geographical and temporal features.
3. Include wind, traffic, and industrial activity.
4. Model PM2.5 forecasting (regression + thresholding).
5. Add satellite imagery features.

## 28 · Production Considerations

- Deploy for real-time AQI monitoring dashboards.
- Trigger health advisories at Unhealthy thresholds.
- Integrate with weather forecasting.
- Monitor sensor calibration drift.
- Provide location-specific predictions.

## 29 · Common Mistakes

1. Not recognizing that PM2.5 largely determines AQI (leakage-like).
2. Ignoring temporal patterns (time of day, season).
3. Using accuracy without per-category analysis.
4. Not considering pollutant measurement uncertainty.
5. Applying a model when simple thresholds suffice.

## 30 · Mini Challenge / Exercises

1. Remove `pm25` — can other pollutants predict AQI category?
2. Try regression on raw PM2.5 and then threshold.
3. Merge Unhealthy_Sensitive + Unhealthy and compare.
4. Analyze the impact of temperature on AQI category.
5. Plot pollutant correlation network.

## 31 · Final Summary / Key Takeaways

1. **PM2.5** is the dominant AQI category driver.
2. **Other pollutants** provide correlated but complementary signal.
3. **Weather** (temperature, humidity) affects pollutant dispersion.
4. **Ordinal classification** fits the natural AQI ordering.
5. **Simple thresholds** may outperform ML for this specific problem.
6. **Real-world AQI** requires continuous monitoring and forecasting.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Air Quality Category Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9919,
    "f1": 0.9919,
    "precision": 0.9919,
    "recall": 0.9919
  },
  "LightGBM": {
    "accuracy": 1.0,
    "f1": 1.0,
    "precision": 1.0,
    "recall": 1.0
  },
  "Logistic Regression": {
    "accuracy": 0.9856,
    "f1": 0.9854,
    "precision": 0.9854,
    "recall": 0.9856
  },
  "FLAML": {
    "accuracy": 1.0,
    "f1": 1.0,
    "precision": 1.0,
    "recall": 1.0
  }
}
